In [37]:
import requests
import simplejson
import pandas as pd
import sys
from copy import deepcopy

In [21]:
root_url = "https://sc-data-dev.emsl.pnl.gov"
samples = requests.get(f"{root_url}/sample?page=1&per_page=1000").json()['samples']

In [22]:
sample_ecoregions = requests.get(f"{root_url}/elastic/metadata").json()

In [25]:
toc_tns = []
pages = sys.maxsize
page = 1
while page <= pages:
    res = requests.get(f"{root_url}/elastic/TOC_TN?page={page}&per_page=100&data=true").json()
    records = res['data']
    pages = res['pages']
    page += 1
    toc_tns.extend(records)

In [27]:
toc_tns_dict = {f"{tt['proposal_id']}_{tt['sampling_set']}_{tt['core_section']}": tt for tt in toc_tns}

In [28]:
toc_tns_dict

{'60846_2_TOP': {'proposal_id': 60846,
  'sampling_set': 2,
  'core_section': 'TOP',
  'type': 'WEOM',
  'toc': 50.2,
  'tn': 3.73,
  'toc_unit': 'mg per kg',
  'tn_unit': 'mg per kg',
  'flag_tn': None,
  'flag_toc': None,
  'flag_tn_avg': None,
  'flag_toc_avg': None,
  'toc_avg': 42.61,
  'tn_avg': 2.23},
 '60846_2_BTM': {'proposal_id': 60846,
  'sampling_set': 2,
  'core_section': 'BTM',
  'type': 'WEOM',
  'toc': 18.55,
  'tn': 1.26,
  'toc_unit': 'mg per kg',
  'tn_unit': 'mg per kg',
  'flag_tn': None,
  'flag_toc': None,
  'flag_tn_avg': None,
  'flag_toc_avg': None,
  'toc_avg': 18.7,
  'tn_avg': 1.43},
 '60846_5_BTM': {'proposal_id': 60846,
  'sampling_set': 5,
  'core_section': 'BTM',
  'type': 'WEOM',
  'toc': 56.63,
  'tn': 3.09,
  'toc_unit': 'mg per kg',
  'tn_unit': 'mg per kg',
  'flag_tn': None,
  'flag_toc': None,
  'flag_tn_avg': None,
  'flag_toc_avg': None,
  'toc_avg': 59.7,
  'tn_avg': 3.2},
 '60846_5_TOP': {'proposal_id': 60846,
  'sampling_set': 5,
  'core_sec

In [11]:
sample_ecoregions_dict = {se['sampling_data']['id']: se['sampling_data'] for se in sample_ecoregions}

In [ ]:
monet_samples = []
for sample in samples:
    sampling_ecoregion = sample_ecoregions_dict[sample['id']]
    has_toc_tn_top = f"{sample['proposal_id']}_{sample['sampling_set']}_TOP" in toc_tns_dict.keys()
    has_toc_tn_bottom = f"{sample['proposal_id']}_{sample['sampling_set']}_BTM" in toc_tns_dict.keys()
    if has_toc_tn_top:
        toc_tn_top = toc_tns_dict[f"{sample['proposal_id']}_{sample['sampling_set']}_TOP"]
    if has_toc_tn_bottom:
        toc_tn_bottom = toc_tns_dict[f"{sample['proposal_id']}_{sample['sampling_set']}_BTM"]

    monet_sample = {
        "ber_data_source": "MONET",
        "coordinates": {
            "latitude": float(sample['latitude']),
            "longitude": float(sample['longitude']),
            "altitude": None,
            "depth": None,
            "elevation": {
                "numeric_value": float(sample['elevation']['value']),
                "unit": sample['elevation']['unit']
            }
        },
        "entity_type": [
            f"sample"
        ],
        "description": None,
        "id": sample['id'],
        "name": f"MONet Core {sample['proposal_id']}_{sample['sampling_set']}",
        "alt_ids": None,
        "alt_names": None,
        "part_of_collection": None,
        "uri": "https://sc-data.emsl.pnnl.gov/monet",
        "properties": [
            {
                "attribute": {
                    "id": "MIXS:0000332",
                    "label": "soil_type"
                },
                "raw_value": sample['soil_metadata']['soil_type']
            },
            {
                "attribute": {
                    "id": "MIXS:0000011",
                    "label": "collection_date"
                },
                "raw_value": sample['collection_date']
            },
            {
                "attribute": {
                    "label": "ecoregion"
                },
                "raw_value": sampling_ecoregion['eco_info']['ecoregion']
            }
        ]
    }
    monet_sample_top = deepcopy(monet_sample)
    monet_sample_bottom = deepcopy(monet_sample)
    monet_sample_top['name'] = f"{monet_sample_top['name']}_TOP"
    monet_sample_bottom['name'] = f"{monet_sample_bottom['name']}_BOTTOM"
    if has_toc_tn_top:
        monet_sample_top['properties'].extend([
            {
                "attribute": {
                    "label": "toc_avg"
                },
                "raw_value": f"{toc_tn_top['toc_avg']} {toc_tn_top['toc_unit']}",
                "unit": toc_tn_top['toc_unit'],
                "numeric_value": toc_tn_top['toc_avg']
            },{
                "attribute": {
                    "label": "tn_avg"
                },
                "raw_value": f"{toc_tn_top['tn_avg']} {toc_tn_top['tn_unit']}",
                "unit": toc_tn_top['tn_unit'],
                "numeric_value": toc_tn_top['tn_avg']
            }

        ])
    if has_toc_tn_bottom:
        monet_sample_bottom['properties'].extend([
            {
                "attribute": {
                    "label": "toc_avg"
                },
                "raw_value": f"{toc_tn_bottom['toc_avg']} {toc_tn_bottom['toc_unit']}",
                "unit": toc_tn_bottom['toc_unit'],
                "numeric_value": toc_tn_bottom['toc_avg']
            },{
                "attribute": {
                    "label": "tn_avg"
                },
                "raw_value": f"{toc_tn_bottom['tn_avg']} {toc_tn_bottom['tn_unit']}",
                "unit": toc_tn_bottom['tn_unit'],
                "numeric_value": toc_tn_bottom['tn_avg']
            }

        ])
    monet_samples.extend([monet_sample_top, monet_sample_bottom])


In [50]:
emsl_samples_df = pd.read_json('emsl_samples.json')

In [51]:
emsl_samples = emsl_samples_df.to_dict(orient='records')

In [52]:
emsl_samples.extend(monet_samples)

In [53]:
with open('data/emsl_00001.json', 'w') as f:
    f.write(simplejson.dumps(emsl_samples, ignore_nan=True))